# Exercise 03 — Workbook: simulate your own protein

This notebook is **deliberately almost empty**, and it is entirely about **your** protein
— the one you picked in ex01's Character Sheet and predicted in ex02.

Its companion, **`ex03_guide.ipynb`**, works one simulation through in full:
2JAC (yeast glutaredoxin-1) in explicit solvent, from raw PDB download to per-residue
flexibility. **Read the guide first.** Then work here, with it
open beside you.

## How to work in this notebook

- **Copy code across from the guide and adapt it.** The setup, minimisation, equilibration
  and MDAnalysis cells are all generic. Reusing them is the point.
- **Structure the notebook however you like.** The sections below are a checklist of what
  has to be here, not a cell-by-cell template.
- **The reasoning is what's graded, not the code.** Every section asks for a written
  conclusion as well as output.
- **Start early.** A simulation that has to reach a stable RMSD is the one part of this
  course you cannot rush the night before.

## Using AI Tools

You may use AI assistants for syntax, debugging, and code snippets. You must supply the
physical reasoning, the justification for your simulation choices, and the critical
evaluation of what came back. **"The AI said so" is not an answer.**

**Note:** the flexibility prompt below is predict-first — you write your prediction down
*before* you run anything. That written prediction is the graded artefact.

---

## 1. Prepare your protein

Most PDB entries are **not** simulation-ready: missing atoms, missing residues,
alternate conformations, non-standard residues. Fix yours before OpenMM sees it.

[PDBFixer](https://github.com/openmm/pdbfixer) is the tool for this. It also has a web
interface if you prefer to click:

```
!pdbfixer YOURPROTEIN.pdb
```

**Report:** what was actually wrong with your structure? Missing residues? A ligand or
cofactor you had to strip? Say what you removed and why — that choice shapes everything
downstream.

## 2. Predict first, before you simulate

**Before running anything:** which region of your protein do you expect to be **most
flexible**, and which most rigid? Name specific residues or regions.

Base it on what you already know about this protein from ex01 and ex02:
- the B-factors and unresolved residues you found in **ex01**
- the pLDDT profile you found in **ex02**

Do those two agree with each other? If they disagree, say so — that is interesting, not
a problem.

> **My prediction (most flexible):**
>
> **My prediction (most rigid):**
>
> **My reasoning, citing my ex01 and ex02 findings:**

## 3. Run the simulation

Use OpenMM, following the guide's recipe. State and justify your choices:

- **Solvent:** explicit water (as the guide does) or vacuum? The guide measures the real
  cost of each: solvation roughly triples the atom count, but it also damps motion
  realistically — the same protein in vacuum shows RMSF more than twice as large. Vacuum
  is faster and is a *different physical system*, not a cheaper route to the same answer.
  Which did you pick, and what does it cost you?
- **Force field**, **integrator**, **timestep**, **temperature**
- **How long** you ran, and why you stopped there

Run until the RMSD and total energy **stabilise**. In real work you would run far longer;
say what you would do with more time.

## 4. Analyse the trajectory

With MDAnalysis, produce and interpret:

- **RMSD of the backbone** vs. time — has it plateaued? If not, your simulation is not
  equilibrated and every number after this is provisional.
- **Total potential energy** vs. time, on shared subplots with the RMSD. Are they
  correlated? Why or why not?
- **Radius of gyration** vs. time — is the protein staying compact, or unfolding?
- **Per-residue RMSF** — which regions actually moved most?

**Then answer:** how did your section 2 prediction hold up? Name the regions you got
right and the ones you got wrong, and say what you think you missed.

If your protein has a hydrophobic core, track the distances between its core side chains
over time. Is the core stable, or does it loosen as the simulation runs?

---

## 5. Simulate the complex, not just the protein

Everything so far has been the **apo** protein — ligand stripped. That is a real
simplification, and this section is where you confront it.

Your protein came with **its ligand already identified** on the Character Sheet in ex01 —
you do not have to work out which of the crystal's small molecules matters. The guide
strips its ligand for an honest reason: the Amber14 force field has parameters for the 20
standard amino acids and not for arbitrary small molecules. A ligand needs its own parameters, generated
by a **small-molecule force field** — GAFF, SMIRNOFF, or the ML-trained Espaloma.

### 5a. What the complex looks like before you simulate anything

Do this first — it needs no force field at all, and it is worth doing even if 5b defeats
you. Load the **original** structure with its ligand still in it and characterise the
binding site:

- Which residues lie within 4 Å of the ligand? Those are your contact residues.
- Which contacts are **hydrogen bonds**, which are **hydrophobic**, which are
  **electrostatic**? (MDAnalysis `distances`/`contacts` will do this; ex04 adds PLIP.)
- Is the ligand **covalently** bound? Check for `LINK` or `SSBOND` records mentioning it
  in the PDB file. A covalent adduct needs entirely different topology handling and is
  out of scope here — if yours is covalent, say so and stop at 5a. (2JAC's GSH has **no**
  such record, which is what makes it a clean non-covalent example — run the same check on
  yours rather than assuming.)
- Cross-check: do the contact residues overlap the **conserved positions from ex01**?

> **Contact residues:**
>
> **Covalent or non-covalent, and how I checked:**

### 5b. Simulating with the ligand present

**This is the one part of the course that needs conda, not pip.** Be aware of what you
are taking on before you start:

- `openff-toolkit` **cannot be installed with pip** — its only PyPI release is *yanked*
  (verified 2026-08-29). GAFF needs AmberTools and Espaloma needs `espaloma`; neither is
  on PyPI either. There is no pip-only route.
- The working path on Colab is **`condacolab`**: `pip install condacolab` →
  `condacolab.install()` → **the kernel restarts** → then
  `mamba install -c conda-forge openff-toolkit`. Budget real time for the solve.
- That is a **second** kernel restart in this course. Keep track of which one you are on.

Then: parametrise the ligand with `openmmforcefields`' template generator, combine it with
the protein force field, solvate the **complex**, and run the same protocol as before.

**If it defeats you, that is a legitimate outcome — document it.** Write down what you
tried, where it failed, and the exact error. A clear account of a tooling wall is worth
full marks here; a silently missing section is not. This is a genuine limitation of the
open-source MD stack in 2026, not a gap in your ability.

### 5c. What the ligand changed

If 5b worked, compare the **holo** (with ligand) and **apo** (without) trajectories:

- Does the binding-site region become **more rigid** when the ligand is present? Compare
  the per-residue RMSF over your 5a contact residues, holo vs apo.
- Does the ligand stay in its crystallographic pose, or drift? Measure its RMSD over time.
- Did anything move that you did not expect?

**Predict before you compare:** a bound ligand usually damps motion in the residues that
grip it. Is that what you see?

> **My prediction:**
>
> **What I found:**

## 6. Compare against your other two structures

You now have three versions of the same protein:

1. the **experimental** structure (ex01)
2. the **AlphaFold** model (ex02)
3. the **last frame** of your simulation (this exercise)

Superimpose them and report the RMSDs between all three pairs. Then interpret:

- Did the simulation drift away from the experimental structure, or stay near it?
- Is the simulated conformation closer to the crystal or to the AlphaFold model?
- **What would each answer mean?** A model that drifts is not automatically wrong, and a
  model that stays put is not automatically right — say what each outcome would tell you
  about the structure, the force field, and the length of your run.

---

## 7. FRONTIER → Block D (Docking)

**Signpost:** you will fully understand *why* this matters in Lectures 10-12, when we
cover docking. You are being asked to reason ahead of the material on purpose.

**The question:** you have just measured, for your own protein, which regions are rigid
and which move. If someone docked a small molecule into one of its pockets, would that
pocket's **real flexibility** make docking **easier or harder** than treating the protein
as a single rigid structure?

**Hint:** think about rigid-body docking versus induced fit — does a pocket that moves
help or hurt an algorithm that assumes the protein does not move at all?

**Low-stakes:** bonus/calibration item. It does not block finishing this exercise, and
being wrong is expected and not penalised.

**Minimum viable answer:** "I predict flexibility here would make docking
[easier / harder] because ___." One defended line is complete.

**Answer it about your own protein, using your own RMSF numbers** — name the pocket or
region you have in mind and quote the RMSF you measured for it. A general answer about
proteins in the abstract is not what is being asked.

> **My prediction:**
>
> **The region and its measured RMSF:**

You will come back to this in ex04 and find out whether you were right.

---

## Before you submit

- [ ] Structure preparation described, including **what you stripped and why**
- [ ] Section 2 prediction **written before** the simulation was run
- [ ] Simulation choices (solvent, force field, length) **justified**, not just stated
- [ ] RMSD shown to have **plateaued** — or explicitly flagged as not equilibrated
- [ ] Every plot has a **written interpretation**, not just an axis label
- [ ] Three-way structural comparison done and interpreted
- [ ] Section 5a contact analysis done (needs no force field — no excuse to skip it)
- [ ] Section 5b attempted, and **its outcome documented** whether or not it worked
- [ ] Section 7 Frontier answered about **your** protein, citing **your** RMSF numbers
- [ ] Anything that didn't work is **described honestly** rather than deleted — a
      documented dead end earns marks, a silently missing section does not
